# Alpha Model - Google Colab Notebook

A complete alpha model pipeline demonstrating factor investing, machine learning enhancement, purged/embargoed cross-validation, ensemble techniques, and profit attribution analysis.

## Overview

This notebook implements a cross-sectional factor strategy with:
- **Factor Engineering**: Momentum, Reversal, Volatility, Skewness, Trend
- **Cross-Sectional Ranking**: Market-neutral z-scoring
- **Factor Analysis**: Information Coefficients & backtesting with transaction costs
- **ML Enhancement**: LightGBM with feature engineering
- **Time-Series CV**: Purged/Embargoed cross-validation
- **Profit Attribution**: Brinson-style factor/regime/model decomposition

## Installation

Run this cell to install dependencies:

In [ ]:
!pip install -q yfinance lightgbm pandas numpy scipy scikit-learn matplotlib

print("Dependencies installed successfully!")

## Configuration

Key parameters for the strategy. Modify these to suit your universe and horizon.

In [ ]:
# Configuration block - easily modifiable
TICKERS = ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'GLD', 'TLT', 'BTC-USD', 'ETH-USD', 'SOL-USD']
START = '2020-01-01'
END = '2024-12-31'
HORIZON = 5  # Forward return horizon in days
N_SPLITS = 4  # Number of CV folds
SEED = 42     # Random seed for reproducibility
TRANSACTION_COST = 0.001  # 0.1% commission per trade
SLIPPAGE = 0.0005         # 0.05% slippage per trade
OOS_START = '2024-06-01'  # Out-of-sample start date
PURGED_EMBARGO = 21       # Days to embargo/purge for CV (1 month)

print("📋 Configuration:")
print(f"   TICKERS: {TICKERS}")
print(f"   START: {START}")
print(f"   END: {END}")
print(f"   HORIZON: {HORIZON}")
print(f"   N_SPLITS: {N_SPLITS}")
print(f"   SEED: {SEED}")
print(f"   TRANSACTION_COST: {TRANSACTION_COST}")
print(f"   SLIPPAGE: {SLIPPAGE}")
print(f"   OOS_START: {OOS_START}")
print(f"   PURGED_EMBARGO: {PURGED_EMBARGO}")

## 1. Data Download

Downloading adjusted close prices for our universe of tickers using Yahoo Finance.

In [ ]:
# Download data
print(f"📥 Downloading data for {len(TICKERS)} tickers from {START} to {END}...")

import yfinance as yf
import pandas as pd
import numpy as np

data_raw = yf.download(
    tickers=TICKERS,
    start=START,
    end=END,
    auto_adjust=True,
    progress=False
)['Close']

# Handle single ticker case
if len(TICKERS) == 1:
    data_raw = data_raw.to_frame()
    data_raw.columns = TICKERS

# Forward fill and drop columns with all NaN
data = data_raw.ffill().dropna(axis=1, how='all')

# Calculate returns
returns = data.pct_change()

print(f"   Loaded {len(data.columns)} assets, {len(data)} trading days")
print(f"   Date range: {data.index[0].date()} to {data.index[-1].date()}")

# Show basic statistics
display(data.describe().T)

## 2. Factor Engineering

Computing five core factors that will be combined into our alpha signal:

1. **Momentum (12-1)**: 12-month return skipping the most recent month (Jegadeesh & Titman, 1993)
2. **Reversal (1-month)**: Negative of 1-month return (short-term mean reversion)
3. **Volatility (60-day)**: 60-day rolling standard deviation of returns (annualized)
4. **Skewness (60-day)**: 60-day rolling skewness of returns
5. **Trend (20-day)**: Slope of daily linear regression over 20 days (trend strength)

All factors are cross-sectionally ranked (z-scored) within each date to create market-neutral signals.

In [ ]:
# Compute factors
print("🔧 Computing factors...")

from scipy import stats

# Initialize factor dataframe
factors = pd.DataFrame(index=data.index, columns=data.columns)

# 1. Momentum (12-1): 12-month return skipping last month
factors['mom_12_1'] = data.pct_change(252).shift(21)  # Approx 12 months - 1 month

# 2. Reversal (1-month): Negative 1-month return
reversal_1m = -returns.rolling(21).mean()  # Approx 1 month
factors['reversal_1m'] = reversal_1m

# 3. Volatility (60-day): 60-day rolling standard deviation
vol_60 = returns.rolling(60).std() * np.sqrt(252)  # Annualized
factors['vol_60'] = vol_60

# 4. Skewness (60-day): 60-day rolling skewness
skew_60 = returns.rolling(60).skew()
factors['skew_60'] = skew_60

# 5. Trend (20-day): Slope of daily linear regression over 20 days
def linreg_slope(series):
    if len(series) < 2:
        return np.nan
    x = np.arange(len(series))
    slope, _, _, _, _ = stats.linregress(x, series)
    return slope

trend_20 = data.rolling(20).apply(linreg_slope, raw=False)
factors['trend_20'] = trend_20

# Cross-sectional ranking (z-score) for each factor
ranked_factors = factors.copy()
for col in factors.columns:
    # Rank within each date (cross-sectional)
    ranked = factors[col].rank(axis=1, pct=True)
    # Convert to z-scores: (rank - 0.5) * sqrt(12) for approx normal distribution
    # For simplicity, we'll use rank - 0.5 giving range [-0.5, 0.5]
    ranked_factors[col] = ranked - 0.5

# Combined factor signal (equal weight)
factor_signal = ranked_factors.mean(axis=1)

print(f"   Computed {len(factors.columns)} factors")
print(f"   Factor signal range: [{factor_signal.min().min():.3f}, {factor_signal.max().max():.3f}]")

# Show factor correlations
factor_corr = ranked_factors.unstack().groupby(level=0).corr().unstack(level=0)
print("\n📊 Factor correlations:")
display(factor_corr.round(2))

## 3. Factor Analysis & Backtest

Analyzing the predictive power of each factor using Information Coefficient (IC)
and running a factor-only backtest with transaction costs.

In [ ]:
# Calculate forward returns
print("⏳ Calculating forward returns...")

# Forward returns aligned with factor calculation date
# Factors calculated at time t predict returns from t+1 to t+HORIZON
forward_returns = data.pct_change(HORIZON).shift(-HORIZON)

print(f"   Forward returns horizon: {HORIZON} days")
print(f"   Forward returns date range: {forward_returns.index[0].date()} to {forward_returns.index[-1].date()}")

# Align factors and forward returns (factors known at time t)
# Shift factors forward by 1 day so they're known at prediction time
factors_for_prediction = factors.shift(1)

# Drop NaN values
combined_data = pd.concat([factors_for_prediction, forward_returns], axis=1).dropna()

if len(combined_data) == 0:
    # Fallback: use same-day alignment for simplicity
    print("   Warning: No overlapping data with shift, using same-day alignment")
    factors_for_prediction = factors
    combined_data = pd.concat([factors, forward_returns], axis=1).dropna()

# Separate factors and returns
factor_data = combined_data.iloc[:, :len(factors.columns)]
return_data = combined_data.iloc[:, len(factors.columns):]

# Calculate Information Coefficients (IC) for each factor
print("\n📊 Calculating Information Coefficients (IC)...")
ic_values = {}
for factor in factors.columns:
    # Calculate time-series average of cross-sectional correlations
    ic_series = []
    for date in factor_data.index:
        # Get factor values for all tickers at this date
        factor_vals = factor_data.loc[date, factors.columns]
        # Get corresponding returns for all tickers at this date
        return_vals = return_data.loc[date, factors.columns]
        # Calculate cross-sectional correlation
        if len(factor_vals) > 1 and not factor_vals.isna().all() and not return_vals.isna().all():
            ic = np.corrcoef(factor_vals.fillna(0), return_vals.fillna(0))[0, 1]
            if not np.isnan(ic):
                ic_series.append(ic)
    
    if ic_series:
        ic_values[factor] = np.mean(ic_series)
    else:
        ic_values[factor] = 0.0

ic_series = pd.Series(ic_values)

print("\n📊 Factor Information Coefficients (IC):")
print("   (Time-series average of cross-sectional IC)")
for factor, ic in ic_series.items():
    print(f"   {factor:10s}: {ic:+.4f} {'✅' if abs(ic) > 0.02 else '⚠️'}")

# Factor returns backtest
print("\n📈 Factor-only backtest (equal weight long/short):")

# For each date, go long top-ranked factors, short bottom-ranked
# We'll use the factor signal directly as position size (market neutral)
factor_position = factor_signal.clip(-1, 1)  # Limit to [-1, 1] for diversification

# Calculate returns: position known at t-1, return from t-1 to t
factor_returns_bt = (factor_position.shift(1) * returns).sum(axis=1)  # Sum across all tickers

# Apply transaction costs
position_changes = factor_position.diff().abs()  # Absolute change in position
transaction_costs = (position_changes * TRANSACTION_COST).sum(axis=1)  # Cost per trade
slippage_costs = (position_changes * SLIPPAGE).sum(axis=1)  # Slippage per trade
factor_returns_net = factor_returns_bt - transaction_costs - slippage_costs

# Calculate cumulative returns
factor_cumulative = (1 + factor_returns_net.fillna(0)).cumprod()

# Calculate performance metrics
total_return = factor_cumulative.iloc[-1] - 1
trading_days = len(factor_returns_net)
annual_return = (1 + total_return) ** (252 / trading_days) - 1 if trading_days > 0 else 0
annual_vol = factor_returns_net.std() * np.sqrt(252) if trading_days > 1 else 0
sharpe = annual_return / annual_vol if annual_vol > 0 else 0

# Sortino ratio (downside deviation)
downside_returns = factor_returns_net[factor_returns_net < 0]
downside_deviation = downside_returns.std() * np.sqrt(252) if len(downside_returns) > 1 else 0
sortino = annual_return / downside_deviation if downside_deviation > 0 else 0

# Calmar ratio (annual return / max drawdown)
rolling_max = factor_cumulative.cummax()
drawdown = (factor_cumulative / rolling_max) - 1
max_drawdown = drawdown.min()
calmar = annual_return / abs(max_drawdown) if max_drawdown != 0 else 0

# Win rate
wins = (factor_returns_net > 0).sum()
total_non_zero = (factor_returns_net != 0).sum()
win_rate = wins / total_non_zero if total_non_zero > 0 else 0

print(f"   Total Return: {total_return:+.2%}")
print(f"   Annual Return: {annual_return:+.2%}")
print(f"   Annual Volatility: {annual_vol:.2%}")
print(f"   Sharpe Ratio: {sharpe:.2f}")
print(f"   Sortino Ratio: {sortino:.2f}")
print(f"   Calmar Ratio: {calmar:.2f}")
print(f"   Max Drawdown: {max_drawdown:.2%}")
print(f"   Win Rate: {win_rate:.2%}")
print(f"   Total Trading Days: {trading_days}")

# Plot factor performance
import matplotlib.pyplot as plt
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Equity curve
ax1.plot(factor_cumulative.index, factor_cumulative.values, linewidth=2, color='darkblue')
ax1.set_title('Factor Model Equity Curve', fontsize=14, fontweight='bold')
ax1.set_ylabel('Cumulative Return')
ax1.grid(True, alpha=0.3)
ax1.legend(['Factor Model'])

# Drawdown
ax2.fill_between(drawdown.index, drawdown.values, 0, color='red', alpha=0.3)
ax2.set_title('Drawdown', fontsize=14, fontweight='bold')
ax2.set_ylabel('Drawdown')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. ML Feature Engineering

Creating machine learning features from price and volume data.

Features include:
- Returns: 5, 10, 20, 60, 120 day returns
- Volatility: 5, 10, 20, 60, 120 day volatility
- Risk-adjusted returns: Sharpe-like ratios for each lookback
- All features are cross-sectionally ranked (z-scored) within each date

In [ ]:
# Create ML features
print("🛠 Creating ML features...")

def create_ml_features(prices):
    """Create ML features from price data."""
    feat = pd.DataFrame(index=prices.index)
    r = prices.pct_change()
    
    # Returns and volatilities for multiple lookbacks
    lookbacks = [5, 10, 20, 60, 120]
    for lb in lookbacks:
        feat[f'ret_{lb}d'] = r.rolling(lb).sum()
        feat[f'vol_{lb}d'] = r.rolling(lb).std() * np.sqrt(252)  # Annualized
        feat[f'sharpe_{lb}d'] = feat[f'ret_{lb}d'] / (feat[f'vol_{lb}d'] + 1e-8)  # Avoid division by zero
    
    # Rank features cross-sectionally (z-score)
    for col in feat.columns:
        ranked = feat[col].rank(axis=1, pct=True)
        feat[col] = ranked - 0.5  # Range [-0.5, 0.5]
    
    return feat.dropna()

ml_features = create_ml_features(data)

print(f"   Created {len(ml_features.columns)} ML features")
print(f"   Feature data shape: {ml_features.shape}")

# Show feature correlations (first 5 features)
if len(ml_features.columns) >= 5:
    feature_corr = ml_features.iloc[:, :5].corr()
    print("\n📊 Sample feature correlations (first 5):")
    display(feature_corr.round(2))

## 5. Time-Series Cross-Validation

Using Purged/Embargoed Cross-Validation to prevent lookahead bias in financial time series.

**Why Purged/Embargoed CV?**
- Standard CV can leak future information into training
- Purging removes samples close to the test period in time
- Embargoing adds a buffer period after the test period
- This ensures no temporal leakage between train and test sets

**Configuration**: 4-fold CV with 21-day purge/embargo

In [ ]:
# Purged/Embargoed Time-Series Cross-Validation
print("🧪 Running Purged/Embargoed Time-Series Cross-Validation...")

# Prepare data for ML: features predict forward returns
print("\n⏳ Preparing ML dataset...")

# Features at time t predict forward returns from t+1 to t+HORIZON
ml_features_lagged = ml_features.shift(1)  # Features known at previous day
forward_returns_ml = data.pct_change(HORIZON).shift(-HORIZON)  # Future returns

# Stack data for sklearn
features_stacked = ml_features_lagged.stack()
returns_stacked = forward_returns_ml.stack()

# Align indices
common_index = features_stacked.index.intersection(returns_stacked.index)
X = features_stacked.loc[common_index]
y = returns_stacked.loc[common_index]

# Remove NaN values
valid_idx = ~(X.isna().any(axis=1) | y.isna())
X = X[valid_idx]
y = y[valid_idx]

print(f"   ML dataset: {X.shape[0]} samples, {X.shape[1]} features")
print(f"   Date range: {X.index.get_level_values(0)[0].date()} to {X.index.get_level_values(0)[-1].date()}")

# Purged/Embargoed Time-Series Cross-Validation
def purged_embargoed_cv(X, y, n_splits=5, purge_days=0, embargo_days=0):
    """Generate train/test indices for purged/embargoed CV."""
    n_samples = len(X)
    indices = np.arange(n_samples)
    test_size = n_samples // (n_splits + 1)  # Approximate equal sizes
    
    test_starts = [i * test_size for i in range(n_splits)]
    test_ends = [(i + 1) * test_size for i in range(n_splits)]
    
    for i in range(n_splits):
        test_start = test_starts[i]
        test_end = test_ends[i]
        
        # Test indices
        test_indices = indices[test_start:test_end]
        
        # Train indices: everything before test period (with embargo) and after test period (with purge)
        train_before_end = max(0, test_start - embargo_days)
        train_after_start = test_end + purge_days
        
        train_indices = np.concatenate([
            indices[0:train_before_end],
            indices[train_after_start:n_samples]
        ])
        
        yield train_indices, test_indices

# Run CV
import lightgbm as lgb
ics = []
models = []
fold_details = []

cv_splitter = purged_embargoed_cv(X, y, n_splits=N_SPLITS, purge_days=PURGED_EMBARGO, embargo_days=PURGED_EMBARGO)

for fold, (train_idx, test_idx) in enumerate(cv_splitter, 1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    # Train LightGBM model
    model = lgb.LGBMRegressor(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=SEED + fold,
        n_jobs=-1,
        verbosity=-1
    )    
    model.fit(X_train, y_train)        # Predict and calculate IC    y_pred = model.predict(X_test)    ic = np.corrcoef(y_pred, y_test)[0, 1] if len(y_pred) > 1 else 0    ics.append(ic)    models.append(model)        fold_details.append({        'fold': fold,        'train_size': len(X_train),        'test_size': len(X_test),        'ic': ic,        'train_date_range': f"{X_train.index.get_level_values(0)[0].date()} to {X_train.index.get_level_values(0)[-1].date()}",        'test_date_range': f"{X_test.index.get_level_values(0)[0].date()} to {X_test.index.get_level_values(0)[-1].date()}"    })        print(f"   Fold {fold}: IC = {ic:+.4f} (Train: {len(X_train)}, Test: {len(X_test)})")print(f"\n📊 Cross-Validation Results:")print(f"   Mean IC: {np.mean(ics):+.4f} ± {np.std(ics):.4f}")print(f"   IC Range: [{np.min(ics):+.4f}, {np.max(ics):+.4f}]")# Show fold detailsfold_df = pd.DataFrame(fold_details)display(fold_df[['fold', 'train_size', 'test_size', 'ic', 'train_date_range', 'test_date_range']])

## 6. Final Model Training & Today's Signal

Training the final model on all available data and generating today's trading signal.

In [ ]:
# Train final model on all data
print("🏗 Training final model on all data...")

final_model = lgb.LGBMRegressor(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1
)
final_model.fit(X, y)

# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\n📊 Top 10 Most Important Features:")
display(feature_importance.head(10))
# Generate today's signal (most recent data point)
print("\n📈 Generating today's trading signal...")
# Get most recent features for each ticker
# We need to get the latest features for each ticker
latest_features = ml_features.groupby(ml_features.index.get_level(1)).last()
# Align with feature names
latest_features_aligned = latest_features.reindex(columns=X.columns, fill_value=0)
# Generate predictions
today_signal_raw = final_model.predict(latest_features_aligned)
today_signal = pd.Series(today_signal_raw, index=latest_features.index)
# Convert to ranked signal (long/short)
today_signal_ranked = today_signal.rank(pct=True) - 0.5  # Range [-0.5, 0.5]
print(f"   Generated signals for {len(today_signal)} tickers")print(f"   Signal range: [{today_signal_ranked.min():+.3f}, {today_signal_ranked.max():+.3f}]")
# Show today's signals
today_signals_df = pd.DataFrame({
    'raw_signal': today_signal,
    'ranked_signal': today_signal_ranked,
    'position': today_signal_ranked.clip(-1, 1)  # Limit position size
}).sort_values('ranked_signal', ascending=False)
display(today_signals_df.head(10))
# Count longs and shorts
n_long = (today_signals_df['ranked_signal'] > 0).sum()
n_short = (today_signals_df['ranked_signal'] < 0).sum()
n_neutral = (today_signals_df['ranked_signal'] == 0).sum()
print(f"\n📊 Signal Summary:")print(f"   Longs:  {n_long:3d} 🟢")print(f"   Shorts: {n_short:3d} 🔴")print(f"   Neutral:{n_neutral:3d} ⚪")

## 8.5 Profit Attribution Analysis

Enhancing our performance analysis with profit attribution to understand what's driving returns.

We'll use the new profit attribution functions to decompose performance by:
- Factors (momentum, value, quality, etc.)
- Market regimes (trend up, trend down, chop)
- Models (in our ensemble)

This helps identify sources of alpha and inform capital allocation decisions.

In [ ]:
# ============================================================
# 8.5 Profit Attribution Analysis
# ============================================================
print("\n📊 Running Complete Profit Attribution Analysis...")

# --- 1. Prepare Data for Attribution Analysis ---
# We'll use the factor signal we already computed
# For attribution, we need to break this down into individual factor returns

print("📊 Recreating individual factors for attribution...")

# Momentum (12-1): 12-month return skipping last month
mom_12_1 = data.pct_change(252).shift(21)
# Reversal (1-month): Negative 1-month return  
reversal_1m = -returns.rolling(21).mean()
# Volatility (60-day): 60-day rolling standard deviation (inverted for quality factor)
vol_60 = returns.rolling(60).std()
quality_factor = 1 / (vol_60 + 1e-8)  # Invert volatility for quality (lower vol = higher quality)
# Value proxy: mean reversion (negative 60-day return)
value_factor = -returns.rolling(60).mean()

# Combine into factor returns DataFrame (assets x factors)
factor_returns = pd.DataFrame({
    'momentum': mom_12_1,
    'reversal': reversal_1m,
    'quality': quality_factor,
    'value': value_factor
})

# Cross-sectionally normalize factors to have mean 0, std 1 each date
factor_returns_norm = factor_returns.copy()
for col in factor_returns.columns:
    factor_returns_norm[col] = factor_returns[col].sub(factor_returns[col].mean(axis=1), axis=0).div(
        factor_returns[col].std(axis=1).replace(0, 1), axis=0)

factor_returns = factor_returns_norm

# Factor weights (equal weight for demo, or we could use ML-derived weights)
factor_weights = pd.DataFrame(
    0.25,  # Equal weight
    index=factor_returns.index,
    columns=factor_returns.columns
)

# --- 2. Calculate Portfolio Returns ---
# Portfolio return = sum(factor_return * factor_weight) across factors for each asset, then average across assets
asset_returns = (factor_returns * factor_weights).sum(axis=1)  # Returns per asset per date
portfolio_returns = asset_returns.mean(axis=1)  # Average return across assets each date

# --- 3. Generate Regime Labels ---
# Use momentum factor to detect regimes
trend_signal = factor_returns['momentum'].rolling(20).mean().fillna(0)
regimes = pd.Series(
    np.where(trend_signal > 0.001, 'trend_up',
            np.where(trend_signal < -0.001, 'trend_down', 'chop')),
    index=factor_returns.index
)

# --- 4. Model Predictions (for ensemble attribution) ---
# Simple models: 
# Model 1: Recent momentum (5-day)
model1_pred = factor_returns['momentum'].rolling(5).mean().shift(1)
# Model 2: Mean reversion (5-day)  
model2_pred = -factor_returns['momentum'].rolling(5).mean().shift(1)

# Normalize predictions
model1_pred = (model1_pred - model1_pred.mean()) / (model1_pred.std() + 1e-8)
model2_pred = (model2_pred - model2_pred.mean()) / (model2_pred.std() + 1e-8)

model_predictions = {
    'momentum_model': model1_pred,
    'mean_reversion_model': model2_pred
}
model_weights = {'momentum_model': 0.6, 'mean_reversion_model': 0.4}

# --- 5. PROFIT ATTRIBUTION FUNCTIONS ---
# Implementing the key attribution functions inline for self-contained execution

def calculate_profit_attribution_by_factor(factor_returns, factor_weights, periods_per_year=252):
    """Calculate profit attribution by factor (Brinson-style)"""
    common_index = factor_returns.index.intersection(factor_weights.index)
    factor_returns = factor_returns.loc[common_index]
    factor_weights = factor_weights.loc[common_index]
    
    if len(common_index) == 0:
        return pd.DataFrame(columns=['factor', 'total_return', 'attribution', 'contribution_pct'])
    
    portfolio_returns = (factor_returns * factor_weights).sum(axis=1)
    total_return = (1 + portfolio_returns).prod() - 1
    
    attribution_data = []
    
    for factor in factor_returns.columns:
        factor_ret = factor_returns[factor]
        avg_weight = factor_weights[factor].mean()
        factor_contribution = (avg_weight * factor_ret.mean()) * len(factor_returns)
        
        attribution_data.append({
            'factor': factor,
            'total_return': (1 + factor_ret).prod() - 1,
            'attribution': factor_contribution,
            'contribution_pct': (factor_contribution / total_return * 100) if total_return != 0 else 0
            })
        
    return pd.DataFrame(attribution_data)

def calculate_profit_attribution_by_regime(returns, regimes, periods_per_year=252):
    """Calculate profit attribution by market regime"""
    common_index = returns.index.intersection(regimes.index)
    returns = returns.loc[common_index]
    regimes = regimes.loc[common_index]
    
    if len(common_index) == 0:
        return pd.DataFrame(columns=['regime', 'total_return', 'annualized_return', 'contribution_pct'])
    
    attribution_data = []
    
    for regime in regimes.unique():
        regime_mask = regimes == regime
        regime_returns = returns[regime_mask]
        
        regime_total_return = (1 + regime_returns).prod() - 1
        regime_annualized = (1 + regime_total_return) ** (periods_per_year / len(regime_returns)) - 1 if len(regime_returns) > 0 else 0
        
        total_return = (1 + returns).prod() - 1
        
        attribution_data.append({
            'regime': regime,
            'total_return': regime_total_return,
            'annualized_return': regime_annualized,
            'contribution_pct': (regime_total_return / total_return * 100) if total_return != 0 else 0
            })
        
    return pd.DataFrame(attribution_data)

def calculate_profit_attribution_by_model(model_predictions, actual_returns, model_weights, periods_per_year=252):
    """Calculate profit attribution by model in ensemble"""
    common_index = actual_returns.index
    for model_name, preds in model_predictions.items():
        common_index = common_index.intersection(preds.index)
    
    model_predictions = {k: v.loc[common_index] for k, v in model_predictions.items()}
    actual_returns = actual_returns.loc[common_index]
    
    if len(common_index) == 0:
        return pd.DataFrame(columns=['model', 'total_return', 'attribution', 'contribution_pct'])
    
    attribution_data = []
    portfolio_returns = pd.Series(0, index=actual_returns.index)
    
    for model_name in model_predictions:
        model_pred = model_predictions[model_name]
        model_weight = model_weights.get(model_name, 0)
        
        model_return = (model_pred * actual_returns).mean() * model_weight
        
        attribution_data.append({
            'model': model_name,
            'total_return': (1 + model_return).prod() - 1 if len(model_return) > 0 else 0,
            'attribution': model_return * len(actual_returns) if len(actual_returns) > 0 else 0,
            'contribution_pct': 0
            })
    
    total_return = sum(d['total_return'] for d in attribution_data)
    for d in attribution_data:
        d['contribution_pct'] = (d['total_return'] / total_return * 100) if total_return != 0 else 0
    
    return pd.DataFrame(attribution_data)

def calculate_factor_half_life(factor_series, max_lag=252):
    """Estimate half-life of a factor using autocorrelation decay"""
    clean = factor_series.dropna()
    if len(clean) < 10:
        return np.nan
    
    # Calculate autocorrelation at different lags
    acfs = []
    for lag in range(1, min(max_lag, len(clean)//4)):
        acf = factor_series.autocorr(lag=lag)
        if np.isnan(acf):
            break
        acfs.append(acf)
    
    if len(acfs) < 2:
        return np.nan
    
    # Fit exponential decay: acf = exp(-lag / half_life)
    lags = np.arange(1, len(acfs) + 1)
    log_acfs = np.log(np.abs(acfs) + 1e-8)
    
    if np.all(np.isnan(log_acfs)):
        return np.nan
    
    # Linear regression: log(acf) = -lag / half_life
    slope, _, _, _, _ = stats.linregress(lags, log_acfs)
    half_life = -1 / slope if slope < 0 else np.nan
    
    return half_life

def calculate_regime_transition_probability(regimes):
    """Calculate regime transition probability matrix"""
    regimes_clean = regimes.dropna()
    if len(regimes_clean) < 2:
        return pd.DataFrame()
    
    # Create transition pairs
    transitions = pd.DataFrame({
        'from': regimes_clean.shift(1),
        'to': regimes_clean
    }).dropna()
    
    if len(transitions) == 0:
        return pd.DataFrame()
    
    # Count transitions
    transition_counts = transitions.groupby(['from', 'to']).size().unstack(fill_value=0)
    
    # Convert to probabilities (row-wise)
    transition_prob = transition_counts.div(transition_counts.sum(axis=1), axis=0)
    
    return transition_prob

# ============================================================
# 8.5.1 Factor Attribution
# ============================================================
print("\n📊 1. Factor Attribution Analysis")
print("-" * 40)

factor_attr = calculate_profit_attribution_by_factor(factor_returns, factor_weights)
display(factor_attr)

# ============================================================
# 8.5.2 Regime Attribution
# ============================================================
print("\n📊 2. Regime Attribution Analysis")
print("-" * 40)

regime_attr = calculate_profit_attribution_by_regime(portfolio_returns, regimes)
display(regime_attr)

# ============================================================
# 8.5.3 Model Attribution
# ============================================================
print("\n📊 3. Model Attribution Analysis")
print("-" * 40)

model_attr = calculate_profit_attribution_by_model(model_predictions, portfolio_returns, model_weights)
display(model_attr)

# ============================================================
# 8.5.4 Factor Half-Life Analysis
# ============================================================
print("\n📊 4. Factor Half-Life Analysis")
print("-" * 40)

print("Factor half-life (estimated decay time):")
for factor in factor_returns.columns:
    half_life = calculate_factor_half_life(factor_returns[factor])
    print(f"{factor:10s}: {half_life:3.0f} days")
# Interpretation
print("\n📊 Interpretation:")print("   - Shows how long a factor's edge persists before decaying")print("   - Helps identify factors that may need replacement or re calibration")print("   - Shorter half-life = faster decay = needs more frequent updating")
# ============================================================
# 8.5.5 Regime Transition Probability Matrix
# ============================================================
print("\n📊 5. Regime Transition Probability Matrix")print("-" * 40)
transition_prob = calculate_regime_transition_probability(regimes)display(transition_prob)
# Interpretationprint("\n📊 Interpretation:")print("   - Shows probability of moving from one regime to another")print("   - Helps improve regime detection models")print("   - Informs expected holding period calculations")print("   - Higher diagonal values = more persistent regimes")

## 8.6 Key Insights from Profit Attribution

The profit attribution analysis provides valuable insights for strategy improvement:

### 🔍 What We Learned

1. **Factor Attribution**: Identifies which factors (momentum, value, quality) are driving returns
2. **Regime Attribution**: Shows how strategy performs in different market environments
3. **Model Attribution**: Quantifies each model's contribution to ensemble performance
4. **Factor Half-Life**: Measures how quickly factor signals decay over time
5. **Regime Transitions**: Shows likelihood of regime changes

### 🎯 How to Use These Insights

- **Capital Allocation**: Increase allocation to high-performing factors/models
- **Risk Management**: Reduce exposure to decaying factors or challenging regimes
- **Model Improvement**: Focus on improving or replacing underperforming models
- **Regime Adaptation**: Adjust strategy parameters based on regime persistence
- **Timely Rebalancing**: Use half-life and transition probabilities to inform rebalancing frequency

---
*Profit attribution completes our performance analysis toolkit, enabling data-driven strategy optimization*